# putEMG — Per-Subject / Per-Class Accuracy Analysis

Deep-dives into **which gestures are hard** and **which subjects are outliers** in the
cross-subject LOSO experiments.

Sections:
1. **Summary from logs** — lightweight, no inference required
2. **Model comparison** — per-subject accuracy across EMG_TCN / EEGNet / ShallowConvNet
3. **Per-class inference** — loads saved checkpoints and collects per-class predictions
4. **Confusion matrices** — per model (aggregated across subjects)
5. **Per-class accuracy heatmap** — subjects × gesture classes
6. **Outlier analysis** — best/worst subjects per gesture

**Prerequisites:** Run `experiments/cross_subject/deep_learning/model.ipynb` for all three models first.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

import torch
import torch.nn as nn

In [ ]:
# ── Shared utilities ──────────────────────────────────────────────────────────
sys.path.append(os.path.abspath('../../'))

import src.deep_learning_models as dlm
from src.emg_loader import load_all_subjects, BCIDataset
from torch.utils.data import DataLoader

In [ ]:
_notebook_dir = os.path.abspath(os.getcwd())

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR       = '/Volumes/KRIS/data/UG_per_subject'
CS_DL_DIR      = os.path.abspath('../cross_subject/deep_learning')
WS_DL_DIR      = os.path.abspath('../within_subject/deep_learning')
RESULTS_DIR    = os.path.join(_notebook_dir, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Gesture names as used in the putEMG dataset (class indices 0-6)
GESTURE_NAMES = ['G1 Open', 'G2 Fist', 'G3 Pinch', 'G6 Ext', 'G7 Flex', 'G8 RadDev', 'G9 UlnDev']
NUM_CLASSES   = len(GESTURE_NAMES)

MODELS_CS = ['EMG_TCN', 'EEGNet', 'ShallowConvNet']   # cross-subject
MODEL_MAP  = {
    'EMG_TCN':        dlm.EMG_TCN,
    'EEGNet':         dlm.EEGNet,
    'ShallowConvNet': dlm.ShallowConvNet,
}

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Cross-subject results dir: {CS_DL_DIR}')

---
## 1. Per-Subject Accuracy Summary (from log files)

In [ ]:
def load_checkpoints(weights_dir):
    # Load all .pt files in weights_dir and return a list of checkpoint dicts.
    ckpts = []
    if not os.path.isdir(weights_dir):
        return ckpts
    for fname in sorted(os.listdir(weights_dir)):
        if not fname.endswith('.pt'):
            continue
        ckpt = torch.load(os.path.join(weights_dir, fname),
                          map_location='cpu', weights_only=False)
        if 'test_acc' in ckpt:
            ckpts.append(ckpt)
    return ckpts


def subject_id_from_mat(mat_name):
    return mat_name.replace('emg_gestures_', '').replace('_U.mat', '')


# Gather results for all three models
model_results = {}   # model_type → list of (subject_id, test_acc)
for mt in MODELS_CS:
    wdir  = os.path.join(CS_DL_DIR, 'weights', mt)
    ckpts = load_checkpoints(wdir)
    model_results[mt] = [(subject_id_from_mat(c['subject']), c['test_acc'] * 100)
                         for c in ckpts]
    n = len(ckpts)
    accs = [c['test_acc'] * 100 for c in ckpts]
    print(f'  {mt:<18} {n:>2} folds  mean={np.mean(accs):.2f}%  std={np.std(accs):.2f}%'
          if accs else f'  {mt:<18} 0 folds  (no checkpoints found)')

In [ ]:
# Per-subject bar chart — all models side by side
all_sids = sorted({sid for sids_accs in model_results.values()
                   for sid, _ in sids_accs})
x        = np.arange(len(all_sids))
n_models = len(MODELS_CS)
w        = 0.25
colors   = ['steelblue', 'darkorange', 'seagreen']

fig, ax = plt.subplots(figsize=(max(14, len(all_sids) * 0.6), 5))

for i, (mt, col) in enumerate(zip(MODELS_CS, colors)):
    acc_map = dict(model_results[mt])
    accs    = [acc_map.get(sid, np.nan) for sid in all_sids]
    offset  = (i - n_models / 2 + 0.5) * w
    bars    = ax.bar(x + offset, accs, w, label=f'{mt} ({np.nanmean(accs):.1f}%)',
                     color=col, alpha=0.85, edgecolor='none')

ax.axhline(50, color='grey', linestyle=':', linewidth=1, label='Chance (50%)')
ax.set_xticks(x)
ax.set_xticklabels(all_sids, rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 105)
ax.set_xlabel('Test Subject')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Cross-Subject LOSO — Per-Subject Accuracy by Model')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'per_subject_comparison.png'), dpi=150)
plt.show()

In [ ]:
# Rank subjects: easiest (highest mean accuracy) → hardest (lowest mean)
sid_means = {}
for sid in all_sids:
    vals = [acc for mt in MODELS_CS
            for s, acc in model_results[mt] if s == sid]
    if vals:
        sid_means[sid] = np.mean(vals)

ranked = sorted(sid_means.items(), key=lambda kv: kv[1], reverse=True)
print(f'{"Rank":>4}  {"Subject":>12}  {"Mean Acc":>9}')
print('-' * 30)
for rank, (sid, acc) in enumerate(ranked, 1):
    flag = '  ← easiest' if rank == 1 else ('  ← hardest' if rank == len(ranked) else '')
    print(f'{rank:>4}  {sid:>12}  {acc:>8.2f}%{flag}')

---
## 2. Per-Class Inference

Loads every saved model checkpoint, runs it on its corresponding test subject, and
collects per-sample predictions. This allows computing confusion matrices and per-class
accuracy — at the cost of running inference on 44 subjects × 3 models.

> **Runtime:** ~10–30 seconds per fold (CPU/MPS inference only, no training).

In [ ]:
def collect_predictions(model_type, subjects_data, cs_dl_dir, device):
    # For every checkpoint of model_type, load the model and run inference
    # on its corresponding test subject.
    # Returns: all_true (N,), all_pred (N,), per_subj list of (sid, true, pred)
    weights_dir = os.path.join(cs_dl_dir, 'weights', model_type)
    if not os.path.isdir(weights_dir):
        print(f'  No weights directory for {model_type} — skipping.')
        return np.array([]), np.array([]), []

    # Build a fast lookup from subject_id → (X, y) for the test set
    subj_lookup = {subject_id_from_mat(name): (X, y)
                   for name, X, y in subjects_data}

    ModelClass = MODEL_MAP[model_type]
    all_true, all_pred, per_subj = [], [], []

    for fname in sorted(os.listdir(weights_dir)):
        if not fname.endswith('.pt'):
            continue
        ckpt = torch.load(os.path.join(weights_dir, fname),
                          map_location='cpu', weights_only=False)
        if 'subject' not in ckpt or 'state_dict' not in ckpt:
            continue

        sid = subject_id_from_mat(ckpt['subject'])
        if sid not in subj_lookup:
            print(f'  [WARN] {sid} not found in loaded subjects — skipping.')
            continue

        X_test, y_test = subj_lookup[sid]

        # Instantiate model and load weights
        model = ModelClass(num_classes=NUM_CLASSES, num_channels=24,
                           dropout_rate=ckpt.get('dropout', 0.1))
        model.load_state_dict(ckpt['state_dict'])
        model.eval().to(device)

        loader = DataLoader(BCIDataset(X_test, y_test), batch_size=32, shuffle=False)

        true_batch, pred_batch = [], []
        with torch.no_grad():
            for X_b, y_b in loader:
                preds = model(X_b.to(device)).argmax(dim=1).cpu()
                pred_batch.append(preds.numpy())
                true_batch.append(y_b.numpy())

        true_np = np.concatenate(true_batch)
        pred_np = np.concatenate(pred_batch)
        all_true.append(true_np)
        all_pred.append(pred_np)
        per_subj.append((sid, true_np, pred_np))
        print(f'  {model_type}  {sid}  acc={np.mean(true_np == pred_np)*100:.1f}%')

    return (np.concatenate(all_true) if all_true else np.array([]),
            np.concatenate(all_pred) if all_pred else np.array([]),
            per_subj)

In [ ]:
print('Loading raw subject data...')
subjects_data = load_all_subjects(DATA_DIR)
print(f'Loaded {len(subjects_data)} subjects.\n')

predictions = {}   # model_type → (all_true, all_pred, per_subj)

for mt in MODELS_CS:
    print(f'--- {mt} ---')
    predictions[mt] = collect_predictions(mt, subjects_data, CS_DL_DIR, device)
    print()

---
## 3. Confusion Matrices (Aggregated)

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, len(MODELS_CS), figsize=(6 * len(MODELS_CS), 5))
if len(MODELS_CS) == 1:
    axes = [axes]

for ax, mt in zip(axes, MODELS_CS):
    all_true, all_pred, _ = predictions[mt]
    if len(all_true) == 0:
        ax.set_title(f'{mt}\n(no data)')
        continue

    cm   = confusion_matrix(all_true, all_pred, normalize='true')
    acc  = np.mean(all_true == all_pred) * 100
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=GESTURE_NAMES, yticklabels=GESTURE_NAMES,
                ax=ax, cbar=False, linewidths=0.3)
    ax.set_title(f'{mt}  (mean {acc:.1f}%)', fontsize=11)
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_ylabel('True', fontsize=9)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.suptitle('Normalised Confusion Matrices — Cross-Subject LOSO (all subjects pooled)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Per-Class Accuracy Breakdown

In [ ]:
# Per-class accuracy for each model
print(f"{'Gesture':<14}", end='')
for mt in MODELS_CS:
    print(f'  {mt:>16}', end='')
print()
print('-' * (14 + 18 * len(MODELS_CS)))

per_class_accs = {}   # model_type → (N_classes,) array

for mt in MODELS_CS:
    all_true, all_pred, _ = predictions[mt]
    if len(all_true) == 0:
        per_class_accs[mt] = np.full(NUM_CLASSES, np.nan)
        continue
    accs = np.array([
        np.mean(all_pred[all_true == c] == c) * 100
        if np.any(all_true == c) else np.nan
        for c in range(NUM_CLASSES)
    ])
    per_class_accs[mt] = accs

for g in range(NUM_CLASSES):
    print(f'{GESTURE_NAMES[g]:<14}', end='')
    for mt in MODELS_CS:
        v = per_class_accs[mt][g]
        print(f'  {v:>15.2f}%', end='')
    print()

print()
print(f"{'Overall':<14}", end='')
for mt in MODELS_CS:
    all_true, all_pred, _ = predictions[mt]
    v = np.mean(all_true == all_pred) * 100 if len(all_true) else np.nan
    print(f'  {v:>15.2f}%', end='')
print()

In [ ]:
# Per-class bar chart
x        = np.arange(NUM_CLASSES)
w        = 0.25
colors   = ['steelblue', 'darkorange', 'seagreen']

fig, ax = plt.subplots(figsize=(11, 5))
for i, (mt, col) in enumerate(zip(MODELS_CS, colors)):
    accs   = per_class_accs.get(mt, np.full(NUM_CLASSES, np.nan))
    offset = (i - len(MODELS_CS) / 2 + 0.5) * w
    ax.bar(x + offset, accs, w, label=mt, color=col, alpha=0.85, edgecolor='none')

ax.axhline(100 / NUM_CLASSES, color='grey', linestyle=':', linewidth=1,
           label=f'Chance ({100/NUM_CLASSES:.1f}%)')
ax.set_xticks(x)
ax.set_xticklabels(GESTURE_NAMES, rotation=25, ha='right', fontsize=9)
ax.set_ylim(0, 105)
ax.set_ylabel('Per-Class Accuracy (%)')
ax.set_title('Cross-Subject LOSO — Per-Class Accuracy by Model')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'per_class_accuracy.png'), dpi=150)
plt.show()

---
## 5. Per-Subject Per-Class Heatmap

In [ ]:
# For the best-performing model, build a subjects × gesture heatmap
best_model = max(MODELS_CS, key=lambda mt: (
    np.mean(predictions[mt][0] == predictions[mt][1]) if len(predictions[mt][0]) else -1
))

_, _, per_subj = predictions[best_model]

if per_subj:
    sids_ordered = [sid for sid, _, _ in per_subj]
    heatmap_data = np.full((len(sids_ordered), NUM_CLASSES), np.nan)

    for row_idx, (sid, true_np, pred_np) in enumerate(per_subj):
        for c in range(NUM_CLASSES):
            mask = true_np == c
            if mask.any():
                heatmap_data[row_idx, c] = np.mean(pred_np[mask] == c) * 100

    fig, ax = plt.subplots(figsize=(10, max(6, len(sids_ordered) * 0.35)))
    im = ax.imshow(heatmap_data, aspect='auto', cmap='RdYlGn',
                   vmin=0, vmax=100, interpolation='nearest')
    plt.colorbar(im, ax=ax, label='Per-class accuracy (%)', shrink=0.6)
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_xticklabels(GESTURE_NAMES, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(len(sids_ordered)))
    ax.set_yticklabels(sids_ordered, fontsize=7)
    ax.set_title(f'{best_model} — Per-Subject Per-Class Accuracy  (Cross-Subject LOSO)',
                 fontsize=12)
    ax.set_xlabel('Gesture Class')
    ax.set_ylabel('Test Subject')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'per_subject_per_class_heatmap.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'No predictions available for {best_model}.')

---
## 6. Outlier Analysis — Best & Worst Subjects per Gesture

In [ ]:
_, _, per_subj = predictions[best_model]

if per_subj:
    # Per-subject per-class accuracy dict: sid → array (NUM_CLASSES,)
    subj_class_acc = {}
    for sid, true_np, pred_np in per_subj:
        subj_class_acc[sid] = np.array([
            np.mean(pred_np[true_np == c] == c) * 100
            if np.any(true_np == c) else np.nan
            for c in range(NUM_CLASSES)
        ])

    print(f'Gesture-level outliers  ({best_model})\n')
    print(f"{'Gesture':<14}  {'Best subject':>16}  {'Best%':>6}  {'Worst subject':>16}  {'Worst%':>6}")
    print('-' * 68)
    for c in range(NUM_CLASSES):
        vals  = {sid: subj_class_acc[sid][c] for sid in subj_class_acc
                 if not np.isnan(subj_class_acc[sid][c])}
        if not vals:
            continue
        best_sid  = max(vals, key=vals.get)
        worst_sid = min(vals, key=vals.get)
        print(f'{GESTURE_NAMES[c]:<14}  {best_sid:>16}  {vals[best_sid]:>5.1f}%  '
              f'{worst_sid:>16}  {vals[worst_sid]:>5.1f}%')

    # Overall outlier subjects
    subj_means = {sid: np.nanmean(arr) for sid, arr in subj_class_acc.items()}
    easiest    = max(subj_means, key=subj_means.get)
    hardest    = min(subj_means, key=subj_means.get)
    print(f'\nOverall easiest subject: {easiest}  ({subj_means[easiest]:.1f}%)')
    print(f'Overall hardest subject: {hardest}  ({subj_means[hardest]:.1f}%)')